In [1]:
!pip install -q langchain langchain-groq langsmith

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.2 MB/s eta 0:00:00


In [11]:
import os

os.environ["GROQ_API_KEY"] = "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"
os.environ["LANGCHAIN_API_KEY"] = "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"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Resume-Screener"

In [22]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2
)

In [23]:
extract_prompt = PromptTemplate.from_template("""
You are an AI Resume Analyzer.

Extract:
1. Skills
2. Experience
3. Tools

Resume:
{resume}
""")

match_prompt = PromptTemplate.from_template("""
Compare the resume with the job description.

Job Description:
{jd}

Resume Details:
{resume_data}

Return:
1. Matching Skills
2. Missing Skills
3. Match Percentage
""")

score_prompt = PromptTemplate.from_template("""
Based on the match analysis:

Assign a Fit Score from 0 to 100.

Explain:
- Why this score was assigned
- Candidate strengths
- Candidate weaknesses

Analysis:
{match_data}
""")

In [24]:
extract_chain = extract_prompt | llm
match_chain = match_prompt | llm
score_chain = score_prompt | llm

In [25]:
jd = """
Hiring for Data Scientist Role.

Required Skills:
Python
SQL
Machine Learning
Deep Learning
Power BI
NLP
"""

In [26]:
strong_resume = """
John Doe
3 years of Data Science Experience.
Skills: Python, SQL, Machine Learning, Deep Learning, NLP, Power BI.
Worked on AI Forecasting Projects.
"""

average_resume = """
Jane Smith
1 year of Data Analyst Experience.
Skills: Python, SQL, Excel, Power BI.
Worked on Dashboards.
"""

weak_resume = """
Mark Taylor
Basic MS Office Knowledge.
Worked in Customer Support.
"""

In [27]:
def screen_resume(resume):

    extracted = extract_chain.invoke({
        "resume": resume
    }).content

    matched = match_chain.invoke({
        "jd": jd,
        "resume_data": extracted
    }).content

    scored = score_chain.invoke({
        "match_data": matched
    }).content

    return scored

In [28]:
print("===== STRONG CANDIDATE =====")
print(screen_resume(strong_resume))

print("\n===== AVERAGE CANDIDATE =====")
print(screen_resume(average_resume))

print("\n===== WEAK CANDIDATE =====")
print(screen_resume(weak_resume))

===== STRONG CANDIDATE =====
Based on the match analysis, I would assign a Fit Score of 100.

This score was assigned because the candidate's resume skills perfectly match the required skills for the job, with all 6 required skills (Python, SQL, Machine Learning, Deep Learning, Power BI, and NLP) being present in the resume. The match percentage is 100%, indicating a perfect match between the candidate's skills and the job requirements.

Candidate strengths:
- The candidate has a strong technical skill set, with expertise in programming languages (Python, SQL), machine learning (Machine Learning, Deep Learning), data visualization (Power BI), and natural language processing (NLP).
- The candidate's skills are highly relevant to the job requirements, indicating a strong potential for success in the role.

Candidate weaknesses:
- None are apparent based on the skills analysis, as the candidate's resume skills perfectly match the required skills for the job. However, it's possible that ot